# Exercise: EKF Navigation for an Autonomous Surface Vehicle

This notebook is the **student version** of the exercise on state estimation for an autonomous surface vehicle (ASV).

## Learning goals
In this exercise, you will:
- generate a synthetic ASV trajectory and simulate sensor measurements
- visualize GNSS, IMU, and compass raw data
- implement **dead reckoning** using IMU only
- implement an **Extended Kalman Filter (EKF)** fusing all three sensors
- evaluate estimator performance during normal operation and **GNSS dropout**

## What this notebook includes

1. A synthetic ASV trajectory generator
2. Sensor simulation for GNSS, IMU, and compass
3. Dead reckoning using IMU only
4. An Extended Kalman Filter (EKF)
5. A GNSS dropout experiment
6. Performance evaluation and plots

## Sensor suite
The ASV is equipped with:
- **GNSS**: noisy position measurements at 1 Hz
- **IMU**: body-frame horizontal acceleration and yaw rate at 10 Hz
- **Compass**: heading measurement at 5 Hz

## State vector
\[
x = [p_N,\ p_E,\ v_N,\ v_E,\ \psi]^T
\]
where \(p_N, p_E\) are north/east position, \(v_N, v_E\) are north/east velocity, and \(\psi\) is heading.

## Assumptions

To keep the first exercise focused on navigation fusion rather than full attitude estimation:

- motion is planar
- roll and pitch are neglected
- the IMU provides horizontal body-frame acceleration
- compass provides heading directly
- GNSS is treated as position-only

## How to work through this notebook
Fill in every cell marked **TODO**. Cells marked *do not modify* provide infrastructure and must run unchanged.


## 1. Imports


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=3, suppress=True)


## 2. Helper functions

`wrap_angle` and `rotmat` are provided. The RMSE helpers are also provided — you will use them in Section 10.


In [ ]:
def wrap_angle(angle):
    """Wrap angle (scalar or array) to [-pi, pi)."""
    return (angle + np.pi) % (2.0 * np.pi) - np.pi


def rotmat(psi):
    """2D body-to-navigation rotation matrix."""
    c = np.cos(psi)
    s = np.sin(psi)
    return np.array([[c, -s],
                     [s,  c]])


def rmse(x, y):
    """Root-mean-square error between two 1-D arrays."""
    return np.sqrt(np.mean((x - y) ** 2))


def rmse_2d(x, y):
    """RMS of Euclidean distance between two (N, 2) position arrays."""
    return np.sqrt(np.mean(np.sum((x - y) ** 2, axis=1)))


## 3. Generate a synthetic ASV trajectory

The trajectory is smooth and marine-like, with moderate turns and speed variation. *Do not modify this cell* — keeping the trajectory fixed ensures your results are comparable to the reference solution.


In [ ]:
# ── Simulation parameters ──────────────────────────────────────────────
dt = 0.1          # IMU sample period [s] (10 Hz)
T  = 600.0        # total duration [s]
t  = np.arange(0.0, T + dt, dt)
N  = len(t)

# ── Ground-truth arrays ─────────────────────────────────────────────────
p_true       = np.zeros((N, 2))   # [pN, pE]
v_true       = np.zeros((N, 2))   # [vN, vE]
a_true_nav   = np.zeros((N, 2))
a_true_body  = np.zeros((N, 2))
psi_true     = np.zeros(N)
r_true       = np.zeros(N)        # yaw rate

# ── Smooth speed and yaw-rate profiles ──────────────────────────────────
speed = (
    1.8
    + 0.4 * np.sin(0.015 * t)
    + 0.2 * np.sin(0.043 * t + 0.8)
)

r_cmd = (
    0.015 * np.sin(0.020 * t)
    + 0.010 * np.sin(0.055 * t + 1.2)
)
r_cmd += np.where((t > 120) & (t < 180),  0.020, 0.0)
r_cmd += np.where((t > 260) & (t < 320), -0.018, 0.0)
r_cmd += np.where((t > 430) & (t < 500),  0.015, 0.0)

# ── Integrate heading ───────────────────────────────────────────────────
for k in range(N - 1):
    r_true[k]       = r_cmd[k]
    psi_true[k + 1] = wrap_angle(psi_true[k] + dt * r_true[k])

# ── Velocity from speed and heading ─────────────────────────────────────
v_true[:, 0] = speed * np.cos(psi_true)   # vN
v_true[:, 1] = speed * np.sin(psi_true)   # vE

# ── Position integration ─────────────────────────────────────────────────
for k in range(N - 1):
    p_true[k + 1] = p_true[k] + dt * v_true[k]

# ── True body-frame acceleration (for IMU simulation) ───────────────────
a_true_nav[1:] = np.diff(v_true, axis=0) / dt
a_true_nav[0]  = a_true_nav[1]
for k in range(N):
    a_true_body[k] = rotmat(psi_true[k]).T @ a_true_nav[k]

print(f"Generated {N} samples over {T:.0f} s")


## 4. Plot the ground-truth trajectory


In [ ]:
# TODO: plot the true trajectory in the North-East plane.
# Hint: p_true[:,1] is East, p_true[:,0] is North — put East on the x-axis.

fig, ax = plt.subplots(figsize=(7, 6))
# ax.plot(...)
ax.set_xlabel("East [m]")
ax.set_ylabel("North [m]")
ax.set_title("Ground-truth ASV trajectory")
ax.axis("equal")
ax.grid(True)
plt.show()


## 5. Simulate sensor measurements

Sensor rates are asynchronous to reflect a realistic fusion problem:
- IMU at 10 Hz (every step)
- GNSS at 1 Hz
- Compass at 5 Hz

*Do not modify this cell.*


In [ ]:
rng = np.random.default_rng(7)

# ── Sensor noise characteristics ────────────────────────────────────────
imu_acc_std   = 0.04                 # m/s²
imu_gyro_std  = np.deg2rad(0.35)     # rad/s
compass_std   = np.deg2rad(4.0)      # rad
gnss_std      = 1.8                  # m

# ── Sensor biases ────────────────────────────────────────────────────────
gyro_bias     = np.deg2rad(0.25)     # rad/s  (constant)
acc_bias_body = np.array([0.03, -0.02])  # m/s²

# ── IMU (10 Hz — every sample) ───────────────────────────────────────────
acc_meas  = a_true_body + acc_bias_body + rng.normal(0.0, imu_acc_std,  size=(N, 2))
gyro_meas = r_true + gyro_bias         + rng.normal(0.0, imu_gyro_std, size=N)

# ── GNSS (1 Hz) ──────────────────────────────────────────────────────────
gnss          = np.full((N, 2), np.nan)
gnss_period   = int(round(1.0 / dt))
gnss_idx      = np.arange(0, N, gnss_period)
gnss[gnss_idx] = p_true[gnss_idx] + rng.normal(0.0, gnss_std, size=(len(gnss_idx), 2))

# ── Compass (5 Hz) ───────────────────────────────────────────────────────
compass             = np.full(N, np.nan)
compass_period      = int(round(0.2 / dt))
compass_idx         = np.arange(0, N, compass_period)
compass[compass_idx] = wrap_angle(
    psi_true[compass_idx] + rng.normal(0.0, compass_std, size=len(compass_idx))
)

# Keep a clean copy for the dropout experiment
gnss_nominal = gnss.copy()

print("Sensors simulated.")
print(f"  GNSS samples:    {np.sum(~np.isnan(gnss[:,0]))}")
print(f"  Compass samples: {np.sum(~np.isnan(compass))}")


## 6. Visualize the raw sensor data

### Questions
- Which sensors provide **absolute** position information?
- Which sensor is most useful for **short-term propagation**?
- Which sensors are most useful for **long-term correction** of drift?


In [ ]:
# TODO: create a 4-panel figure showing all sensor signals.
#
# Suggested layout (4 subplots, sharex=True):
#   ax[0] — IMU body-frame acceleration (ax, ay) vs time
#   ax[1] — IMU yaw rate vs time
#   ax[2] — Compass heading (dots at compass_idx) and psi_true vs time
#   ax[3] — GNSS position scatter (East on x-axis, North on y-axis)
#
# Hint for axes [3]: use ax[3].plot(gnss[:,1], gnss[:,0], '.')
#   (column 1 = East, column 0 = North)

fig, ax = plt.subplots(4, 1, figsize=(10, 12), sharex=False)

# ax[0]: acceleration
# ax[1]: yaw rate
# ax[2]: compass + truth
# ax[3]: GNSS scatter (not vs time — use a separate non-sharex layout if preferred)

plt.tight_layout()
plt.show()


## 7. Dead reckoning baseline

Implement dead reckoning by integrating the IMU only. This shows the classic drift problem that motivates sensor fusion.

Wrap the heading after each propagation step.


In [ ]:
def run_dead_reckoning(acc_meas, gyro_meas, dt, p0, v0, psi0):
    """Propagate state from IMU only. Returns x_dr of shape (N, 5).

    State layout: [pN, pE, vN, vE, psi]
    """
    n = len(gyro_meas)
    x_dr    = np.zeros((n, 5))
    x_dr[0] = np.array([p0[0], p0[1], v0[0], v0[1], psi0])

    for k in range(n - 1):
        pN, pE, vN, vE, psi = x_dr[k]

        # TODO 1: rotate body-frame acceleration to navigation frame
        # a_nav = rotmat(psi) @ acc_meas[k]

        # TODO 2: propagate heading
        # psi_next = wrap_angle(psi + dt * gyro_meas[k])

        # TODO 3: propagate velocity
        # v_next = ...

        # TODO 4: propagate position (use velocity at step k, not k+1)
        # p_next = ...

        # TODO 5: store in x_dr[k+1]
        pass

    return x_dr


# Initialize from the true initial state to isolate estimator drift
x_dr = run_dead_reckoning(
    acc_meas=acc_meas,
    gyro_meas=gyro_meas,
    dt=dt,
    p0=p_true[0],
    v0=v_true[0],
    psi0=psi_true[0],
)


## 8. Plot dead reckoning drift


In [ ]:
# TODO: plot the dead reckoning trajectory against ground truth and GNSS.
# Use East on the x-axis and North on the y-axis.

fig, ax = plt.subplots(figsize=(8, 6))
# ax.plot(p_true[:,1],    p_true[:,0],    label='Ground truth', linewidth=2)
# ax.plot(gnss[:,1],      gnss[:,0],      '.', alpha=0.5, label='GNSS')
# ax.plot(x_dr[:,1],      x_dr[:,0],      label='Dead reckoning')
ax.set_xlabel("East [m]")
ax.set_ylabel("North [m]")
ax.set_title("Dead reckoning drift")
ax.axis("equal")
ax.grid(True)
ax.legend()
plt.show()

# TODO: print dead-reckoning position and heading RMSE
# Hint: use rmse_2d(x_dr[:,0:2], p_true) for position
# Hint: use rmse(wrap_angle(x_dr[:,4] - psi_true), 0) or equivalently
#        np.sqrt(np.mean(wrap_angle(x_dr[:,4] - psi_true)**2)) for heading


## 9. EKF model

### Prediction model
\[
p_{k+1} = p_k + \Delta t\, v_k
\]
\[
v_{k+1} = v_k + \Delta t\, R(\psi_k)\, a_k
\]
\[
\psi_{k+1} = \psi_k + \Delta t\, r_k
\]

### Measurements
GNSS: \(z_{\mathrm{GNSS}} = [p_N,\, p_E]^T + v\)

Compass: \(z_\psi = \psi + v\)

The model is nonlinear because body-frame acceleration is rotated by the current heading estimate. The Jacobian \(F\) therefore has a non-trivial \(\partial v / \partial \psi\) column:
\[
\frac{\partial}{\partial \psi}[R(\psi)\,a] =
\begin{bmatrix}-\sin\psi & -\cos\psi \\ \cos\psi & -\sin\psi\end{bmatrix} a
\]


In [ ]:
def predict_ekf(x, P, a_body, r_meas, dt, Q):
    """EKF prediction step.

    Parameters
    ----------
    x      : state vector [pN, pE, vN, vE, psi]
    P      : 5x5 covariance matrix
    a_body : 2-vector of body-frame acceleration
    r_meas : scalar yaw-rate measurement
    dt     : time step
    Q      : 5x5 process noise covariance

    Returns x_pred, P_pred
    """
    p   = x[0:2]
    v   = x[2:4]
    psi = x[4]

    # TODO 1: rotate body-frame acceleration to navigation frame

    # TODO 2: propagate mean state


    # TODO 3: build Jacobian F = df/dx
    # F is 5x5 identity plus corrections for the v->p and psi->v couplings.

    # TODO 4: propagate covariance

    return x_pred, P_pred


def update_gnss(x, P, z, R_gnss):
    """EKF update step using a GNSS position measurement z = [pN, pE].

    Use the Joseph stabilized covariance form for numerical robustness:
      IKH   = I - K @ H
      P_upd = IKH @ P @ IKH.T + K @ R_gnss @ K.T

    Returns x_upd, P_upd, y (innovation), S (innovation covariance)
    """
    H = np.array([
        [1, 0, 0, 0, 0],
        [0, 1, 0, 0, 0],
    ], dtype=float)

    # TODO 1: predicted measurement:
    # TODO 2: innovation:
    # TODO 3: innovation covariance:
    # TODO 4: Kalman gain:
    # TODO 5: updated state:
    # TODO 6: wrap heading:
    # TODO 7: updated covariance (Joseph form)

    return x_upd, P_upd, y, S


def update_compass(x, P, z, R_compass):
    """EKF update step using a compass heading measurement z (scalar).

    IMPORTANT: wrap the innovation to handle the +-pi discontinuity:
      y = np.array([wrap_angle(z - x[4])])

    Use the Joseph stabilized covariance form.

    Returns x_upd, P_upd, y (1-element array), S (1x1 matrix)
    """
    H = np.array([[0, 0, 0, 0, 1]], dtype=float)

    # TODO 1: innovation with wrapping

    # TODO 2: Joseph form covariance update

    return x_upd, P_upd, y, S


## 10. Run the EKF


In [ ]:
# ── Initial state and covariance ─────────────────────────────────────────
# x0 is the exact true initial state, so P0 should reflect near-zero uncertainty.
x0 = np.array([
    p_true[0, 0],
    p_true[0, 1],
    v_true[0, 0],
    v_true[0, 1],
    psi_true[0],
], dtype=float)

# TODO: tune P0 to get good initial performance
P0 = np.diag([
    0.0,              # pN  (effectively exact)
    0.0,              # pE
    0.0,              # vN
    0.0,              # vE
    0.0,              # psi (effectively exact)
])

# ── Process and measurement noise ────────────────────────────────────────
# TODO: tune Q to get good performance.
# Starting values are provided; experiment by changing them and re-running.
#
# Note: position receives NO independent process noise (it is driven only by
# velocity), so Q[0,0] = Q[1,1] = 0. Only velocity and heading are directly
# perturbed by noise.
Q = np.diag([
    0.0,                    # pN  — not directly noise-driven
    0.0,                    # pE  — not directly noise-driven
    0.0,                    # vN
    0.0,                    # vE
    0.0,                    # psi
])

R_gnss    = np.diag([gnss_std**2, gnss_std**2])
R_compass = np.array([[compass_std**2]])


def run_ekf(acc_meas, gyro_meas, gnss, compass, dt, x0, P0, Q, R_gnss, R_compass):
    """Run the EKF over the full dataset.

    Correct loop structure (avoids off-by-one):
      - store x0 at index 0 before any prediction
      - for k in range(N-1): predict with u[k], update with z[k+1]

    Returns
    -------
    x_hist       : (N, 5) estimated states
    P_hist       : (N, 5, 5) covariance history
    innov_gnss   : (N, 2) GNSS innovations (NaN when no update)
    innov_compass: (N,) compass innovations (NaN when no update)
    nis_gnss     : (N,) Normalized Innovation Squared for GNSS
    nis_compass  : (N,) Normalized Innovation Squared for compass
    """
    N = len(gyro_meas)
    x_hist        = np.zeros((N, 5))
    P_hist        = np.zeros((N, 5, 5))
    innov_gnss    = np.full((N, 2), np.nan)
    innov_compass = np.full(N, np.nan)
    nis_gnss      = np.full(N, np.nan)
    nis_compass   = np.full(N, np.nan)

    x = x0.copy()
    P = P0.copy()

    # Store initial state before any propagation
    x_hist[0] = x
    P_hist[0] = P

    for k in range(N - 1):
        # TODO 1: predict using IMU at time k

        # TODO 2: GNSS update if measurement available at k+1

        # TODO 3: compass update if measurement available at k+1

        # TODO 4: store

        pass

    return x_hist, P_hist, innov_gnss, innov_compass, nis_gnss, nis_compass


x_ekf, P_ekf, innov_gnss, innov_compass, nis_gnss, nis_compass = run_ekf(
    acc_meas=acc_meas,
    gyro_meas=gyro_meas,
    gnss=gnss,
    compass=compass,
    dt=dt,
    x0=x0,
    P0=P0,
    Q=Q,
    R_gnss=R_gnss,
    R_compass=R_compass,
)


## 11. Compare truth, GNSS, dead reckoning, and EKF


In [ ]:
# TODO: plot all four trajectories in the North-East plane.
# Use East on the x-axis, North on the y-axis.

fig, ax = plt.subplots(figsize=(9, 7))
# ax.plot(p_true[:,1],  p_true[:,0],  label='Ground truth', linewidth=2)
# ax.plot(gnss[:,1],    gnss[:,0],    '.', alpha=0.4, label='GNSS')
# ax.plot(x_dr[:,1],   x_dr[:,0],    label='Dead reckoning')
# ax.plot(x_ekf[:,1],  x_ekf[:,0],   label='EKF', linewidth=2)
ax.set_xlabel("East [m]")
ax.set_ylabel("North [m]")
ax.set_title("Trajectory comparison")
ax.axis("equal")
ax.grid(True)
ax.legend()
plt.show()


## 12. Performance metrics


In [ ]:
# TODO: compute and print position and heading RMSE for dead reckoning and EKF.

pos_rmse_dr  = rmse_2d(x_dr[:,  0:2], p_true)
pos_rmse_ekf = rmse_2d(x_ekf[:, 0:2], p_true)

head_rmse_dr  = np.rad2deg(np.sqrt(np.mean(wrap_angle(x_dr[:,  4] - psi_true) ** 2)))
head_rmse_ekf = np.rad2deg(np.sqrt(np.mean(wrap_angle(x_ekf[:, 4] - psi_true) ** 2)))

print("Performance summary")
print("-------------------")
print(f"Dead reckoning position RMSE: {pos_rmse_dr:.3f} m")
print(f"EKF position RMSE:            {pos_rmse_ekf:.3f} m")
print(f"Dead reckoning heading RMSE:  {head_rmse_dr:.3f} deg")
print(f"EKF heading RMSE:             {head_rmse_ekf:.3f} deg")


## 13. Error and innovation plots

These diagnostics help evaluate and tune the filter:
- large or growing errors → model mismatch or poor tuning
- innovations that are too large/erratic → filter is too confident (`R` too small)
- innovations that are too smooth/small → filter is under-confident (`Q` too small)


In [ ]:
# TODO: create a 3-panel figure showing position and heading errors over time.

pos_err_dr  = x_dr[:,  0:2] - p_true
pos_err_ekf = x_ekf[:, 0:2] - p_true
head_err_dr  = np.rad2deg(wrap_angle(x_dr[:,  4] - psi_true))
head_err_ekf = np.rad2deg(wrap_angle(x_ekf[:, 4] - psi_true))

fig, ax = plt.subplots(3, 1, figsize=(11, 10), sharex=True)

# ax[0]: North position error for DR and EKF
# ax[1]: East position error for DR and EKF
# ax[2]: Heading error for DR and EKF

plt.tight_layout()
plt.show()


# TODO: create a 3-panel innovation plot.

fig, ax = plt.subplots(3, 1, figsize=(11, 10), sharex=True)

# ax[0]: innov_gnss[:, 0]  (GNSS North innovation)
# ax[1]: innov_gnss[:, 1]  (GNSS East innovation)
# ax[2]: innov_compass in degrees

plt.tight_layout()
plt.show()


## 14. NIS consistency check

The **Normalized Innovation Squared (NIS)** validates whether the filter noise parameters are well-tuned. For a well-tuned filter:
- GNSS NIS should average near **2** (chi-squared, 2 degrees of freedom)
- Compass NIS should average near **1** (chi-squared, 1 degree of freedom)

A consistently high mean NIS means the filter is **over-confident** (Q too small or R too small). A consistently low mean NIS means it is **under-confident**.


In [ ]:
# TODO: plot NIS for GNSS and compass and print their means.

fig, ax = plt.subplots(2, 1, figsize=(11, 6), sharex=True)

# ax[0]: plot nis_gnss vs t, add horizontal lines at 2.0 (expected) and nanmean
# ax[1]: plot nis_compass vs t, add horizontal lines at 1.0 and nanmean

plt.tight_layout()
plt.show()

print(f"GNSS    NIS mean: {np.nanmean(nis_gnss):.3f}  (ideal ~2.0)")
print(f"Compass NIS mean: {np.nanmean(nis_compass):.3f}  (ideal ~1.0)")


## 15. GNSS dropout experiment

Remove GNSS measurements for a 30-second window and re-run the EKF. Observe how the filter behaves when it must rely solely on IMU propagation and compass corrections.

**Note:** the dropout window (250–280 s) intentionally overlaps the strong turn manoeuvre at 260–320 s. The degraded performance therefore reflects both the GNSS loss and the heading change; these effects are not independent.

### Questions
- How quickly does position error grow during the outage?
- Does the heading remain bounded? Why?
- Which sensor is most important during the outage?


In [ ]:
gnss_dropout = gnss_nominal.copy()

drop_start = 250.0
drop_end   = 280.0
drop_mask  = (t >= drop_start) & (t <= drop_end)
gnss_dropout[drop_mask] = np.nan

x_ekf_drop, P_ekf_drop, innov_gnss_drop, innov_compass_drop, _, _ = run_ekf(
    acc_meas=acc_meas,
    gyro_meas=gyro_meas,
    gnss=gnss_dropout,
    compass=compass,
    dt=dt,
    x0=x0,
    P0=P0,
    Q=Q,
    R_gnss=R_gnss,
    R_compass=R_compass,
)

pos_err_drop = np.linalg.norm(x_ekf_drop[:, 0:2] - p_true, axis=1)
print(f"Maximum position error during GNSS dropout: {np.max(pos_err_drop[drop_mask]):.3f} m")


## 16. GNSS dropout plots


In [ ]:
# TODO: create two comparison plots.

# Plot 1: trajectory — nominal EKF vs dropout EKF vs ground truth
fig, ax = plt.subplots(figsize=(9, 7))
# ax.plot(p_true[:,1],        p_true[:,0],        label='Ground truth', linewidth=2)
# ax.plot(x_ekf[:,1],         x_ekf[:,0],         label='EKF nominal')
# ax.plot(x_ekf_drop[:,1],    x_ekf_drop[:,0],    label='EKF with GNSS dropout')
ax.set_xlabel("East [m]")
ax.set_ylabel("North [m]")
ax.set_title("Effect of GNSS dropout")
ax.axis("equal")
ax.grid(True)
ax.legend()
plt.show()


# Plot 2: position error and heading error vs time, with dropout window shaded
fig, ax = plt.subplots(2, 1, figsize=(11, 8), sharex=True)

# ax[0]: position error magnitude for nominal and dropout EKF
#         ax[0].axvspan(drop_start, drop_end, alpha=0.2, label='Dropout window')
# ax[1]: heading error for nominal and dropout EKF
#         ax[1].axvspan(drop_start, drop_end, alpha=0.2)

plt.tight_layout()
plt.show()


## 17. Interpretation

Write short answers to the discussion questions below.

1. **Why does dead reckoning drift** even when IMU noise is small?
2. **Why does the EKF strongly reduce position drift** compared to dead reckoning?
3. **What is the effect of increasing Q?** What about decreasing it too much?
4. **What is the effect of reducing R_gnss too much?** What does the filter do?
5. **What happens during GNSS dropout?** Which sensor limits how fast the error grows?
6. **What would change** if you added gyro bias as a 6th state?


**Your written answers here.**


## 18. Optional extensions

- add gyro bias to the EKF state (6-state filter)
- add accelerometer bias states to the EKF (8-state filter)
- inspect the NIS plots and re-tune Q and R until NIS means are close to ideal
- test the effect of reducing the compass update rate to 1 Hz
- initialize the filter with a wrong heading and observe convergence


In [ ]:
# Optional work area
